# Experiment 4 — Conflicting-Gender ε Test + Two Follow-Up Controls

**Context:** Every prior attempt at measuring ε (the capacity cost of shared
representations across languages) failed because the two synthetic languages
never actually *contended* for the same parameters — they had compatible,
non-conflicting strategies that could coexist for free.

**This design's core idea:** two languages share the exact same noun
vocabulary, but each has its own arbitrary grammatical gender for every noun
(like German *Sonne* (f.) vs. French *soleil* (m.) — same referent,
contradictory gender). The overlap between the two gender tables is a
controllable knob: `overlap=1.0` means the tables agree on everything (no
conflict possible — this is the ε=0 control); `overlap=0.0` means they
disagree on every single noun (maximal conflict).

**First-pass result (already run in chat, reproduced here for continuity):**
at very low capacity, the `overlap=1.0` control solves both languages by
`n_embd=4`, while `overlap=0.5` and `overlap=0.0` both need `n_embd=5`. This
is a real ε-like threshold shift — but two things need checking before
trusting it:

1. **Concern 1 — no gradient.** `overlap=0.5` and `overlap=0.0` gave the
   *same* threshold instead of an intermediate one. Is there a real
   monotonic relationship between conflict severity and required capacity,
   or does the effect saturate as soon as there's *any* conflict? This
   notebook runs a denser overlap grid with more seeds to check.

2. **Concern 2 — is it just table size?** Maybe the effect is simply "more
   independent facts to memorize needs more capacity" — a raw
   information-content effect that has nothing to do with the *conflict*
   specifically. This notebook adds a **disjoint-vocabulary control**: 16
   total nouns (8 exclusive to each language, never shared), matched to have
   the same total information content as the `overlap=0.0` condition, but
   with **zero possibility of cross-language conflict** (no noun ever
   appears in both languages, so no embedding ever needs to serve two
   masters). If this control solves at a *lower* capacity than the
   shared-vocab max-conflict condition, that's real evidence the conflict
   itself — not just the amount of information — is what costs capacity.

Run all cells top to bottom. The sweeps are the slow part; each config takes
roughly 5-20 seconds depending on your hardware, and the notebook prints
progress as it goes so you can gauge how long it'll take. Feel free to
increase `SWEEP_SEEDS` and `SWEEP_STEPS` at the top of each experiment if you
want tighter confidence — the defaults here are set to *reasonable*, not
*maximal*.


## 0. Setup

In [1]:

import os
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import random
import statistics
import json
from pathlib import Path
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


## 1. Grammar Engine — Shared-Vocabulary Conflicting Gender

Same nouns across both languages. Each language has its own arbitrary
gender table. `overlap` controls what fraction of nouns the two tables
agree on.


In [2]:

N_NOUNS = 8
NOUNS = [f"n{i}" for i in range(N_NOUNS)]

BOS_A, BOS_B = "BOS_A", "BOS_B"
EOS = "EOS"
DET_M, DET_F = "DET_M", "DET_F"

VOCAB = [BOS_A, BOS_B, EOS, DET_M, DET_F] + NOUNS
TOK2IDX = {tok: i for i, tok in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)

DET_M_IDX = TOK2IDX[DET_M]
DET_F_IDX = TOK2IDX[DET_F]


def make_gender_tables(overlap: float, seed: int = 0):
    """Construct gA and gB gender tables with a controlled overlap fraction.

    overlap=1.0 -> tables identical (epsilon=0 control)
    overlap=0.0 -> tables maximally disagree (every noun flipped)
    """
    rng = random.Random(seed)
    n_agree = round(overlap * N_NOUNS)

    genders_A = ["M"] * (N_NOUNS // 2) + ["F"] * (N_NOUNS // 2)
    rng.shuffle(genders_A)
    gA = dict(zip(NOUNS, genders_A))

    gB = dict(gA)
    noun_order = list(NOUNS)
    rng.shuffle(noun_order)
    n_flip = N_NOUNS - n_agree
    for noun in noun_order[:n_flip]:
        gB[noun] = "F" if gA[noun] == "M" else "M"

    return gA, gB


def tokenize(tokens):
    return [TOK2IDX[t] for t in tokens]


def sample_sentence(rng, gA, gB, language=None):
    lang = language if language else ("A" if rng.random() < 0.5 else "B")
    noun = NOUNS[rng.randint(0, N_NOUNS - 1)]
    gender = gA[noun] if lang == "A" else gB[noun]
    det = DET_M if gender == "M" else DET_F
    bos = BOS_A if lang == "A" else BOS_B
    tokens = [bos, noun, det, EOS]
    meta = {"language": lang, "noun": noun, "gender": gender}
    return tokens, meta


def audit_tables(overlap, seed=0):
    gA, gB = make_gender_tables(overlap, seed)
    agree = sum(1 for n in NOUNS if gA[n] == gB[n])
    print(f"overlap={overlap}: actual agreement {agree}/{N_NOUNS} = {agree/N_NOUNS:.3f}")
    for n in NOUNS:
        mark = "agree" if gA[n] == gB[n] else "CONFLICT"
        print(f"  {n}: gA={gA[n]} gB={gB[n]}  [{mark}]")
    return gA, gB

print("=== Sanity check: gender tables at a few overlap values ===")
for ov in [0.0, 0.5, 1.0]:
    audit_tables(ov)
    print()


=== Sanity check: gender tables at a few overlap values ===
overlap=0.0: actual agreement 0/8 = 0.000
  n0: gA=F gB=M  [CONFLICT]
  n1: gA=M gB=F  [CONFLICT]
  n2: gA=F gB=M  [CONFLICT]
  n3: gA=M gB=F  [CONFLICT]
  n4: gA=M gB=F  [CONFLICT]
  n5: gA=M gB=F  [CONFLICT]
  n6: gA=F gB=M  [CONFLICT]
  n7: gA=F gB=M  [CONFLICT]

overlap=0.5: actual agreement 4/8 = 0.500
  n0: gA=F gB=M  [CONFLICT]
  n1: gA=M gB=F  [CONFLICT]
  n2: gA=F gB=F  [agree]
  n3: gA=M gB=M  [agree]
  n4: gA=M gB=F  [CONFLICT]
  n5: gA=M gB=F  [CONFLICT]
  n6: gA=F gB=F  [agree]
  n7: gA=F gB=F  [agree]

overlap=1.0: actual agreement 8/8 = 1.000
  n0: gA=F gB=F  [agree]
  n1: gA=M gB=M  [agree]
  n2: gA=F gB=F  [agree]
  n3: gA=M gB=M  [agree]
  n4: gA=M gB=M  [agree]
  n5: gA=M gB=M  [agree]
  n6: gA=F gB=F  [agree]
  n7: gA=F gB=F  [agree]



## 2. Model + Training Code

Same tiny 2-layer transformer architecture throughout. `n_embd` is the
capacity knob we sweep. `n_head=1` (not 2) — this matters: with `n_head=2`
on very small `n_embd`, `head_dim` gets too small to be meaningful. All
sweeps below use `n_layer=2, n_head=1`.


In [3]:

PAD_IDX = VOCAB_SIZE
FULL_VOCAB_SIZE = VOCAB_SIZE + 1
SEQ_LEN = 4  # BOS noun DET EOS


def generate_batch(batch_size, rng, gA, gB, language=None):
    all_ids, all_targets, metas = [], [], []
    for _ in range(batch_size):
        tokens, meta = sample_sentence(rng, gA, gB, language)
        ids = tokenize(tokens)
        all_ids.append(ids[:-1])
        all_targets.append(ids[1:])
        metas.append(meta)
    input_ids = torch.tensor(all_ids, dtype=torch.long, device=DEVICE)
    targets = torch.tensor(all_targets, dtype=torch.long, device=DEVICE)
    return input_ids, targets, metas


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_head, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        att = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.proj(y)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer, n_head, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd, padding_idx=PAD_IDX)
        self.pos_emb = nn.Embedding(max_seq_len, n_embd)
        self.blocks = nn.ModuleList([TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, x):
        B, T = x.shape
        h = self.tok_emb(x) + self.pos_emb(torch.arange(T, device=x.device))
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_f(h))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_model(n_embd, seed, overlap, n_steps=4000, batch_size=64,
                 lr=3e-4, n_layer=2, n_head=1, eval_every=None):
    if eval_every is None:
        eval_every = n_steps  # only eval at the end by default

    torch.manual_seed(seed)
    rng_train = random.Random(seed)
    gA, gB = make_gender_tables(overlap=overlap, seed=0)  # tables FIXED across all runs/seeds

    model = TinyTransformer(FULL_VOCAB_SIZE, n_embd, n_layer, n_head, SEQ_LEN).to(DEVICE)
    n_params = model.count_parameters()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps, eta_min=lr / 10)

    history = {"n_embd": n_embd, "seed": seed, "overlap": overlap, "n_params": n_params, "eval": []}

    for step in range(1, n_steps + 1):
        model.train()
        input_ids, targets, _ = generate_batch(batch_size, rng_train, gA, gB)
        logits = model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, FULL_VOCAB_SIZE), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if step % eval_every == 0 or step == n_steps:
            rng_eval = random.Random(seed + 10000)  # FIXED eval set every checkpoint
            eval_result = evaluate(model, rng_eval, gA, gB)
            eval_result["step"] = step
            history["eval"].append(eval_result)

    return model, history, gA, gB


def evaluate(model, rng_eval, gA, gB, n_batches=16, batch_size=64):
    model.eval()
    correct = {"A": 0, "B": 0}
    total = {"A": 0, "B": 0}
    correct_conflict = {"A": 0, "B": 0}
    total_conflict = {"A": 0, "B": 0}
    conflict_nouns = {n for n in NOUNS if gA[n] != gB[n]}

    with torch.no_grad():
        for _ in range(n_batches):
            input_ids, targets, metas = generate_batch(batch_size, rng_eval, gA, gB)
            logits = model(input_ids)
            det_logits = logits[:, 1, :]
            preds = torch.where(det_logits[:, DET_M_IDX] > det_logits[:, DET_F_IDX],
                                torch.tensor(DET_M_IDX, device=DEVICE),
                                torch.tensor(DET_F_IDX, device=DEVICE))
            gt = targets[:, 1]
            for i, m in enumerate(metas):
                lang = m["language"]
                is_correct = int(preds[i].item() == gt[i].item())
                correct[lang] += is_correct
                total[lang] += 1
                if m["noun"] in conflict_nouns:
                    correct_conflict[lang] += is_correct
                    total_conflict[lang] += 1

    def sd(a, b):
        return a / b if b > 0 else float("nan")

    return {
        "acc_A": sd(correct["A"], total["A"]),
        "acc_B": sd(correct["B"], total["B"]),
        "acc_A_conflict": sd(correct_conflict["A"], total_conflict["A"]),
        "acc_B_conflict": sd(correct_conflict["B"], total_conflict["B"]),
        "n_conflict_nouns": len(conflict_nouns),
    }


def min_conflict_acc(final_eval):
    """The metric we track: worst-case accuracy on conflict nouns across languages.
    If there are no conflict nouns (overlap=1.0), falls back to overall accuracy.
    """
    if final_eval["n_conflict_nouns"] > 0:
        return min(final_eval["acc_A_conflict"], final_eval["acc_B_conflict"])
    return min(final_eval["acc_A"], final_eval["acc_B"])

print("Training code loaded.")


Training code loaded.


## 3. Quick Smoke Test

Verifies the pipeline runs end to end before committing to the full sweeps.


In [4]:

_, h, gA, gB = train_model(n_embd=8, seed=0, overlap=0.0, n_steps=3000)
final = h["eval"][-1]
print(f"overlap=0.0, n_embd=8: acc_A_conflict={final['acc_A_conflict']:.3f}  "
      f"acc_B_conflict={final['acc_B_conflict']:.3f}")
assert min_conflict_acc(final) > 0.9, "Smoke test failed — check environment/dependencies"
print("Smoke test passed.")


overlap=0.0, n_embd=8: acc_A_conflict=1.000  acc_B_conflict=1.000
Smoke test passed.


## 4. Experiment A — Denser Overlap × Capacity Grid (Concern 1)

**Question:** Is there a real monotonic relationship between conflict
severity (`1 - overlap`) and the capacity needed to solve both languages, or
does the effect saturate as soon as there's *any* conflict at all?

**What to look for:** if the theory is right, `N*` (the smallest `n_embd`
that reliably solves both languages) should shift *smoothly* as `overlap`
decreases from 1.0 to 0.0 — not jump straight from "easy" to "maximally
hard" at the first sign of conflict. Increase `SWEEP_SEEDS` if the result
still looks noisy (seed variance was substantial at tight capacity in
earlier runs — some seeds collapse to a winner-take-all outcome where one
language reaches ~1.0 and the other stays at chance, which is itself an
interesting phenomenon but adds noise to the *average*).


In [ ]:

# ---- Configuration (adjust as needed) ----
SWEEP_SEEDS_A = 6          # seeds per (overlap, n_embd) cell
SWEEP_STEPS_A = 5000       # training steps per run
OVERLAP_GRID = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0]
CAPACITY_GRID_A = [2, 3, 4, 5, 6, 8]
# --------------------------------------------

results_A = {}

for overlap in OVERLAP_GRID:
    for n_embd in CAPACITY_GRID_A:
        accs = []
        for seed in range(SWEEP_SEEDS_A):
            _, h, gA, gB = train_model(n_embd=n_embd, seed=seed, overlap=overlap,
                                       n_steps=SWEEP_STEPS_A)
            accs.append(min_conflict_acc(h["eval"][-1]))
        results_A[(overlap, n_embd)] = {
            "mean": statistics.mean(accs),
            "std": statistics.stdev(accs) if len(accs) > 1 else 0.0,
            "all": accs,
        }
        print(f"overlap={overlap:.3f} n_embd={n_embd:>2d}: "
              f"mean_min_acc={results_A[(overlap,n_embd)]['mean']:.3f} "
              f"(+/-{results_A[(overlap,n_embd)]['std']:.3f})")

with open("results_experiment_A.json", "w") as f:
    json.dump({f"{k[0]}_{k[1]}": v for k, v in results_A.items()}, f, indent=2)
print("\nSaved to results_experiment_A.json")


overlap=0.000 n_embd= 2: mean_min_acc=0.462 (+/-0.046)
overlap=0.000 n_embd= 3: mean_min_acc=0.665 (+/-0.134)
overlap=0.000 n_embd= 4: mean_min_acc=0.874 (+/-0.206)
overlap=0.000 n_embd= 5: mean_min_acc=0.918 (+/-0.145)
overlap=0.000 n_embd= 6: mean_min_acc=1.000 (+/-0.000)
overlap=0.000 n_embd= 8: mean_min_acc=1.000 (+/-0.000)
overlap=0.125 n_embd= 2: mean_min_acc=0.428 (+/-0.024)
overlap=0.125 n_embd= 3: mean_min_acc=0.549 (+/-0.228)
overlap=0.125 n_embd= 4: mean_min_acc=0.809 (+/-0.236)
overlap=0.125 n_embd= 5: mean_min_acc=0.978 (+/-0.055)
overlap=0.125 n_embd= 6: mean_min_acc=1.000 (+/-0.000)
overlap=0.125 n_embd= 8: mean_min_acc=1.000 (+/-0.000)
overlap=0.250 n_embd= 2: mean_min_acc=0.330 (+/-0.024)
overlap=0.250 n_embd= 3: mean_min_acc=0.579 (+/-0.251)


In [ ]:

# Plot: accuracy heatmap across (overlap, n_embd)
fig, ax = plt.subplots(figsize=(8, 5))
import numpy as np

overlap_vals = OVERLAP_GRID
embd_vals = CAPACITY_GRID_A
heat = np.zeros((len(overlap_vals), len(embd_vals)))
for i, ov in enumerate(overlap_vals):
    for j, ne in enumerate(embd_vals):
        heat[i, j] = results_A[(ov, ne)]["mean"]

im = ax.imshow(heat, aspect="auto", cmap="RdYlGn", vmin=0.4, vmax=1.0)
ax.set_xticks(range(len(embd_vals)))
ax.set_xticklabels(embd_vals)
ax.set_yticks(range(len(overlap_vals)))
ax.set_yticklabels([f"{ov:.3f}" for ov in overlap_vals])
ax.set_xlabel("n_embd (capacity)")
ax.set_ylabel("overlap (1.0 = no conflict, 0.0 = max conflict)")
ax.set_title("Min conflict-noun accuracy across capacity x overlap")
plt.colorbar(im, ax=ax, label="mean min accuracy")

# Annotate cells
for i in range(len(overlap_vals)):
    for j in range(len(embd_vals)):
        ax.text(j, i, f"{heat[i,j]:.2f}", ha="center", va="center",
                color="black", fontsize=8)

plt.tight_layout()
plt.savefig("experiment_A_heatmap.png", dpi=150)
plt.show()
print("\nLook at how the 'green zone' (high accuracy) boundary moves left-to-right")
print("as you go down the rows (higher overlap = less conflict).")
print("A smooth diagonal boundary = real gradient. A boundary that jumps")
print("immediately at overlap<1.0 and doesn't move further = saturation.")


## 5. Experiment B — Disjoint-Vocabulary Control (Concern 2)

**Question:** Is the extra capacity needed under conflict actually about the
*conflict* — or is it simply that there's more total information to
memorize (16 independent (language, noun) facts vs. 8)?

**Design:** 16 total nouns, split into two disjoint sets of 8 — one set
used only in Language A sentences, the other only in Language B sentences.
Each noun gets an arbitrary gender (balanced M/F). This has the **same
total information content** as the `overlap=0.0` shared-vocabulary
condition (16 independent facts to memorize either way), but **zero
possibility of conflict**, because no noun's embedding is ever asked to
mean two different things.

**Prediction:** if the earlier result was really about *conflict* (and not
just raw memorization load), this disjoint control should reach ceiling
accuracy at a **lower** `n_embd` than the shared-vocabulary `overlap=0.0`
condition — because here, each noun's embedding only ever has to encode one
fact, with no cross-language interference to resolve.


In [ ]:

DISJOINT_NOUNS_A = [f"da{i}" for i in range(8)]
DISJOINT_NOUNS_B = [f"db{i}" for i in range(8)]

DISJOINT_VOCAB = [BOS_A, BOS_B, EOS, DET_M, DET_F] + DISJOINT_NOUNS_A + DISJOINT_NOUNS_B
DTOK2IDX = {t: i for i, t in enumerate(DISJOINT_VOCAB)}
DISJOINT_VOCAB_SIZE = len(DISJOINT_VOCAB)
D_PAD_IDX = DISJOINT_VOCAB_SIZE
D_FULL_VOCAB_SIZE = DISJOINT_VOCAB_SIZE + 1


def make_disjoint_genders(seed=0):
    rng = random.Random(seed)
    gA = dict(zip(DISJOINT_NOUNS_A, rng.sample(["M"] * 4 + ["F"] * 4, 8)))
    gB = dict(zip(DISJOINT_NOUNS_B, rng.sample(["M"] * 4 + ["F"] * 4, 8)))
    return gA, gB


def sample_disjoint(rng, gA, gB, language=None):
    lang = language if language else ("A" if rng.random() < 0.5 else "B")
    if lang == "A":
        noun = DISJOINT_NOUNS_A[rng.randint(0, 7)]
        gender = gA[noun]
    else:
        noun = DISJOINT_NOUNS_B[rng.randint(0, 7)]
        gender = gB[noun]
    det = DET_M if gender == "M" else DET_F
    bos = BOS_A if lang == "A" else BOS_B
    tokens = [bos, noun, det, EOS]
    return tokens, {"language": lang, "noun": noun, "gender": gender}


def tokenize_disjoint(tokens):
    return [DTOK2IDX[t] for t in tokens]


def generate_batch_disjoint(batch_size, rng, gA, gB):
    all_ids, all_targets, metas = [], [], []
    for _ in range(batch_size):
        tokens, meta = sample_disjoint(rng, gA, gB)
        ids = tokenize_disjoint(tokens)
        all_ids.append(ids[:-1])
        all_targets.append(ids[1:])
        metas.append(meta)
    input_ids = torch.tensor(all_ids, dtype=torch.long, device=DEVICE)
    targets = torch.tensor(all_targets, dtype=torch.long, device=DEVICE)
    return input_ids, targets, metas


DET_M_IDX_D = DTOK2IDX[DET_M]
DET_F_IDX_D = DTOK2IDX[DET_F]


def train_disjoint(n_embd, seed, n_steps=4000, n_layer=2, n_head=1, batch_size=64, lr=3e-4):
    torch.manual_seed(seed)
    rng_train = random.Random(seed)
    gA, gB = make_disjoint_genders(seed=0)  # fixed tables across seeds

    model = TinyTransformer(D_FULL_VOCAB_SIZE, n_embd, n_layer, n_head, SEQ_LEN).to(DEVICE)
    # NOTE: TinyTransformer's embedding padding_idx was set to PAD_IDX (the shared-vocab
    # constant) at class-definition time -- for the disjoint vocab we pass D_FULL_VOCAB_SIZE
    # as vocab_size, so padding_idx=PAD_IDX still refers to an unused slot as long as
    # PAD_IDX < D_FULL_VOCAB_SIZE, which it is not guaranteed to be. Rebuild cleanly instead:
    model.tok_emb = nn.Embedding(D_FULL_VOCAB_SIZE, n_embd, padding_idx=D_PAD_IDX).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_steps, eta_min=lr / 10)

    for step in range(n_steps):
        model.train()
        input_ids, targets, _ = generate_batch_disjoint(batch_size, rng_train, gA, gB)
        logits = model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, D_FULL_VOCAB_SIZE), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    rng_eval = random.Random(seed + 10000)
    correct = {"A": 0, "B": 0}
    total = {"A": 0, "B": 0}
    with torch.no_grad():
        for _ in range(16):
            input_ids, targets, metas = generate_batch_disjoint(64, rng_eval, gA, gB)
            logits = model(input_ids)
            det_logits = logits[:, 1, :]
            preds = torch.where(det_logits[:, DET_M_IDX_D] > det_logits[:, DET_F_IDX_D],
                                torch.tensor(DET_M_IDX_D, device=DEVICE),
                                torch.tensor(DET_F_IDX_D, device=DEVICE))
            gt = targets[:, 1]
            for i, m in enumerate(metas):
                correct[m["language"]] += int(preds[i].item() == gt[i].item())
                total[m["language"]] += 1

    accA = correct["A"] / total["A"]
    accB = correct["B"] / total["B"]
    return min(accA, accB), accA, accB, model.count_parameters()

print("Disjoint-control code loaded.")


In [ ]:

# ---- Configuration ----
SWEEP_SEEDS_B = 6
SWEEP_STEPS_B = 5000
CAPACITY_GRID_B = [2, 3, 4, 5, 6, 8]
# ------------------------

results_B = {}
for n_embd in CAPACITY_GRID_B:
    mins = []
    for seed in range(SWEEP_SEEDS_B):
        min_acc, accA, accB, params = train_disjoint(n_embd=n_embd, seed=seed, n_steps=SWEEP_STEPS_B)
        mins.append(min_acc)
    results_B[n_embd] = {"mean": statistics.mean(mins), "std": statistics.stdev(mins), "all": mins, "params": params}
    print(f"[DISJOINT CONTROL] n_embd={n_embd:>2d} params={params:>5d}: "
          f"mean_min_acc={results_B[n_embd]['mean']:.3f} (+/-{results_B[n_embd]['std']:.3f})")

with open("results_experiment_B.json", "w") as f:
    json.dump(results_B, f, indent=2)
print("\nSaved to results_experiment_B.json")


In [ ]:

# Direct comparison plot: shared-vocab max-conflict (overlap=0.0) vs disjoint control
# (both have the same total information content: 16 independent (lang,noun)->gender facts)

fig, ax = plt.subplots(figsize=(8, 5))

conflict_means = [results_A[(0.0, ne)]["mean"] for ne in CAPACITY_GRID_A]
conflict_stds = [results_A[(0.0, ne)]["std"] for ne in CAPACITY_GRID_A]
control_means = [results_B[ne]["mean"] for ne in CAPACITY_GRID_B]
control_stds = [results_B[ne]["std"] for ne in CAPACITY_GRID_B]
epsilon0_means = [results_A[(1.0, ne)]["mean"] for ne in CAPACITY_GRID_A]

ax.errorbar(CAPACITY_GRID_A, conflict_means, yerr=conflict_stds, marker="o",
           label="Shared vocab, overlap=0.0 (max conflict)", color="crimson", capsize=3)
ax.errorbar(CAPACITY_GRID_B, control_means, yerr=control_stds, marker="s",
           label="Disjoint vocab (matched info, zero conflict)", color="steelblue", capsize=3)
ax.plot(CAPACITY_GRID_A, epsilon0_means, marker="^", linestyle="--",
       label="Shared vocab, overlap=1.0 (epsilon=0 control)", color="seagreen")

ax.axhline(0.9, color="gray", linestyle=":", alpha=0.5, label="0.90 threshold")
ax.set_xlabel("n_embd (capacity)")
ax.set_ylabel("mean min accuracy (worst-case across languages)")
ax.set_title("Does conflict cost MORE capacity than matched-information memorization alone?")
ax.legend()
ax.set_ylim(0.3, 1.05)
plt.tight_layout()
plt.savefig("experiment_B_comparison.png", dpi=150)
plt.show()

print("\nIf the crimson line (real conflict) sits clearly to the RIGHT of the")
print("blue line (same info, no conflict) -- i.e. needs a bigger n_embd to reach")
print("the same accuracy -- that's evidence the conflict itself costs capacity,")
print("not just the amount of information.")
print()
print("If the crimson and blue lines are roughly on top of each other, the earlier")
print("result was likely just a raw memorization-capacity effect, and the")
print("'representational competition' framing needs rethinking.")


## 6. Summary Table

A compact readout of the key comparison points, for pasting into notes or a
draft.


In [ ]:

print(f"{'Condition':<45s} {'N* (first n_embd with mean_acc>=0.90)':>40s}")
print("-" * 87)

def find_nstar(means_by_embd, threshold=0.90):
    for ne in sorted(means_by_embd.keys()):
        if means_by_embd[ne] >= threshold:
            return ne
    return None

eps0 = {ne: results_A[(1.0, ne)]["mean"] for ne in CAPACITY_GRID_A}
eps_max = {ne: results_A[(0.0, ne)]["mean"] for ne in CAPACITY_GRID_A}
disjoint = {ne: results_B[ne]["mean"] for ne in CAPACITY_GRID_B}

print(f"{'Shared vocab, overlap=1.0 (eps=0 control)':<45s} {str(find_nstar(eps0)):>40s}")
print(f"{'Shared vocab, overlap=0.0 (max conflict)':<45s} {str(find_nstar(eps_max)):>40s}")
print(f"{'Disjoint vocab (matched info, no conflict)':<45s} {str(find_nstar(disjoint)):>40s}")

print()
print("Intermediate overlap values (from Experiment A):")
for ov in OVERLAP_GRID:
    means = {ne: results_A[(ov, ne)]["mean"] for ne in CAPACITY_GRID_A}
    print(f"  overlap={ov:.3f}: N* = {find_nstar(means)}")
